# Stage 1B on Colab

The reported Stage 1B results come from a local RTX 3050 run (see `docs/stage_1b_report.md`).
**This notebook has not been executed.** It runs the same commands as the local run, with data and
checkpoints on Google Drive so a dropped session can resume. Choose a GPU runtime first.
It needs about 30 GB on Drive.

MUCS is CC BY-SA 4.0. Keep data, weights and credentials out of the Git repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%env MUCS_DATA_ROOT=/content/drive/MyDrive/mucs
%env MUCS_HF_HOME=/content/drive/MyDrive/mucs/hf_cache
%env PYTHONUTF8=1
!git clone --branch stage-1b-baseline https://github.com/shlok165/hindi-english-code-switching-asr.git
%cd hindi-english-code-switching-asr
!pip install -q -r env/requirements-colab.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Data, tokenizer and checks

Downloads train, test and blind (about 8.8 GB), checks the gzip CRCs, and carves a 4-hour
validation set out of the training recordings. Test and blind are never split or merged.

In [ ]:
!python -m src.data.download --only test blind train
!python -m src.data.manifest --config configs/data.yaml
!python -m src.models.tokenizer --config configs/data.yaml
!python -m src.models.verify_encoder --config configs/model_him4200.yaml
!python -m pytest tests -q
!python -m src.training.smoke_test

## Training

The config is the one used for the reported run: batch of one utterance, 8-step accumulation,
FP16, gradient checkpointing, 38,000 updates. On a 16 GB T4 it needs no change. `--resume auto`
continues from the newest checkpoint after a disconnect.

In [ ]:
!python -m src.training.train --config configs/train_baseline.yaml --resume auto

## KenLM

The Python binding builds from source on Linux. `build_binary` and `lmplz` need Boost, which
Colab can install. The LM is built from training transcripts only.

In [ ]:
!apt-get -qq install -y build-essential cmake libboost-program-options-dev libboost-system-dev libboost-thread-dev libboost-test-dev zlib1g-dev libbz2-dev liblzma-dev
!git clone --depth 1 https://github.com/kpu/kenlm.git /content/kenlm
!cmake -S /content/kenlm -B /content/kenlm/build -DCMAKE_BUILD_TYPE=Release && cmake --build /content/kenlm/build --target lmplz build_binary -j2
!pip install -q kenlm==0.3.0 pygtrie hypothesis && pip install -q --no-deps pyctcdecode==0.5.0
import os
os.environ['PATH'] = '/content/kenlm/build/bin:' + os.environ['PATH']

## Evaluation

Runs the same script as the local run. It tunes the decoder on val, writes the winner to
`configs/decode.yaml`, and only then decodes test and blind. Results go to `results/` in the clone,
so copy that folder to Drive before the session ends.

In [ ]:
!bash scripts/evaluate_baseline.sh data/checkpoints/baseline_rtx3050/best
!cp -r results /content/drive/MyDrive/mucs/results_stage1b